# 03 — Attention Analysis

Research-style analysis of attention patterns in Mythos-150M.
Requires a trained checkpoint at `checkpoints/150m_v2/best.pt`.

Topics:
1. Attention head visualization (heatmaps per layer)
2. GQA group structure — how KV heads serve multiple Q heads
3. Positional bias in attention patterns
4. Entropy of attention distributions (head specialization)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path

from mythos.core.transformer import Mythos, ModelConfig
from mythos.training.checkpoint import load_checkpoint

CHECKPOINT = Path('../checkpoints/150m_v2/best.pt')
TOKENIZER  = Path('../data/debug/tokenizer/tokenizer.json')

torch.manual_seed(0)
device = torch.device('cpu')

In [ ]:
# Load model
model, config = load_checkpoint(str(CHECKPOINT))
model = model.to(device).eval()
print(f'Loaded: {model.get_num_params()/1e6:.1f}M params')
print(f'Layers: {config.n_layers}, Q heads: {config.n_heads}, KV heads: {config.n_kv_heads}')

## 1. Attention Weight Extraction

We hook into attention layers to capture the raw attention weights before the output projection.

In [ ]:
from tokenizers import Tokenizer

try:
    tokenizer = Tokenizer.from_file(str(TOKENIZER))
    prompt = 'The transformer architecture uses self-attention to model'
    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], device=device)
    tokens = enc.tokens
    print(f'Tokenized: {tokens}')
except:
    # Fallback: synthetic token ids for visualization
    print('Tokenizer not found — using synthetic input for visualization')
    seq_len = 12
    input_ids = torch.randint(0, config.vocab_size, (1, seq_len), device=device)
    tokens = [f't{i}' for i in range(seq_len)]

In [ ]:
# Hook to capture attention weights
attention_weights = {}

def make_hook(layer_idx):
    def hook(module, args, output):
        # Re-compute attention weights for visualization (not cached)
        x = args[0]
        cos_hook, sin_hook = args[1], args[2]
        B, T, _ = x.shape
        q = module.wq(x).view(B, T, module.n_heads,    module.head_dim).transpose(1, 2)
        k = module.wk(x).view(B, T, module.n_kv_heads, module.head_dim).transpose(1, 2)
        from mythos.core.rope import apply_rope
        q, k = apply_rope(q, k, cos_hook, sin_hook)
        # Expand KV heads for GQA
        if module.n_rep > 1:
            k = k.repeat_interleave(module.n_rep, dim=1)
        scale = module.head_dim ** -0.5
        attn = torch.softmax((q @ k.transpose(-2, -1)) * scale, dim=-1)
        attention_weights[layer_idx] = attn.detach().cpu()
    return hook

hooks = []
for i, layer in enumerate(model.layers):
    h = layer.attention.register_forward_hook(make_hook(i))
    hooks.append(h)

with torch.no_grad():
    _ = model(input_ids)

for h in hooks:
    h.remove()

print(f'Captured attention from {len(attention_weights)} layers')
print(f'Shape: {next(iter(attention_weights.values())).shape}  (B, n_heads, T, T)')

## 2. Attention Heatmaps (select layers)

In [ ]:
# Visualize attention patterns for first, middle, and last layer
n_layers = len(model.layers)
selected_layers = [0, n_layers // 2, n_layers - 1]
n_heads_show = min(4, config.n_heads)

fig = plt.figure(figsize=(16, 4 * len(selected_layers)))
fig.patch.set_facecolor('#0f172a')
gs = gridspec.GridSpec(len(selected_layers), n_heads_show, figure=fig, hspace=0.4, wspace=0.3)

for row, layer_idx in enumerate(selected_layers):
    attn = attention_weights[layer_idx][0]  # (n_heads, T, T)
    T = attn.shape[-1]
    tok_labels = tokens[:T] if len(tokens) >= T else [f't{i}' for i in range(T)]
    
    for head in range(n_heads_show):
        ax = fig.add_subplot(gs[row, head])
        ax.set_facecolor('#1e293b')
        
        im = ax.imshow(attn[head].numpy(), cmap='viridis', aspect='auto', vmin=0, vmax=attn[head].max().item())
        
        ax.set_title(f'Layer {layer_idx} · Head {head}', color='#94a3b8', fontsize=9)
        if head == 0:
            ax.set_yticks(range(T))
            ax.set_yticklabels(tok_labels, fontsize=7, color='#64748b')
        else:
            ax.set_yticks([])
        ax.set_xticks(range(T))
        ax.set_xticklabels(tok_labels, fontsize=7, color='#64748b', rotation=45, ha='right')
        for spine in ax.spines.values():
            spine.set_edgecolor('#334155')

fig.suptitle('Attention Patterns — Mythos-150M', color='#e2e8f0', fontsize=14, y=0.98)
plt.savefig('../assets/attention_heatmaps.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 3. Attention Entropy — Head Specialization

Entropy of the attention distribution measures how "spread out" a head's attention is.
- High entropy → attends broadly (global context heads)
- Low entropy → attends sharply (local / positional heads)

In [ ]:
def attn_entropy(attn_weights):
    """Mean entropy of attention distribution across sequence positions."""
    # attn_weights: (n_heads, T, T)
    eps = 1e-9
    H = -(attn_weights * (attn_weights + eps).log()).sum(-1)  # (n_heads, T)
    return H.mean(-1)  # (n_heads,)

# Compute entropy per layer per head
all_entropies = []
for layer_idx in range(len(model.layers)):
    attn = attention_weights[layer_idx][0]  # (n_heads, T, T)
    H = attn_entropy(attn)
    all_entropies.append(H.numpy())

entropy_matrix = np.array(all_entropies)  # (n_layers, n_heads)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')

im = ax.imshow(entropy_matrix.T, aspect='auto', cmap='plasma', interpolation='nearest')
ax.set_xlabel('Layer', color='#94a3b8', fontsize=11)
ax.set_ylabel('Head', color='#94a3b8', fontsize=11)
ax.set_title('Attention Entropy per Head per Layer\n(high = broad attention, low = focused)', 
             color='#e2e8f0', fontsize=12)
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values():
    spine.set_edgecolor('#334155')

cbar = plt.colorbar(im, ax=ax)
cbar.ax.tick_params(colors='#94a3b8')
cbar.set_label('Entropy (nats)', color='#94a3b8')

plt.tight_layout()
plt.savefig('../assets/attention_entropy.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print(f'Most focused head: Layer {entropy_matrix.argmin() // config.n_heads}, Head {entropy_matrix.argmin() % config.n_heads} (entropy={entropy_matrix.min():.3f})')
print(f'Most diffuse head: Layer {entropy_matrix.argmax() // config.n_heads}, Head {entropy_matrix.argmax() % config.n_heads} (entropy={entropy_matrix.max():.3f})')